In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

In [2]:
df = pd.read_csv("energydata.csv")
print("Duplicated: ", df.duplicated().sum())
print(df.isna().sum())
df.head()

Duplicated:  0
date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [3]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df = df.drop(['date'], axis=1)
print(df[['year', 'month', 'day', 'hour', 'minute']])

       year  month  day  hour  minute
0      2016      1   11    17       0
1      2016      1   11    17      10
2      2016      1   11    17      20
3      2016      1   11    17      30
4      2016      1   11    17      40
...     ...    ...  ...   ...     ...
19730  2016      5   27    17      20
19731  2016      5   27    17      30
19732  2016      5   27    17      40
19733  2016      5   27    17      50
19734  2016      5   27    18       0

[19735 rows x 5 columns]


In [4]:
X = df.drop(["Appliances"], axis=1)
y = df["Appliances"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [5]:
pipeline_auto = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor())
])

parameter_grid = {
    "model__n_neighbors": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    "model__weights": ['uniform', 'distance'],
    "model__metric": ['euclidean', 'manhattan'],
    "model__algorithm": ['auto', 'brute', 'ball_tree', 'kd_tree'],
    "model__leaf_size": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
}

kf = KFold(n_splits=5, shuffle=True)

scores = ['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_root_mean_squared_error']

grid_search = GridSearchCV(pipeline_auto, parameter_grid, cv=kf, scoring=scores, refit='neg_mean_squared_error')

grid_search.fit(X_train, y_train)

print("Best Parameters")
print(grid_search.best_params_)
print()
print("Best Cross-Validation Accuracy")
print(grid_search.best_score_)

results = pd.DataFrame(grid_search.cv_results_)
results['mean_test_MAE'] = -results['mean_test_neg_mean_absolute_error']
results['mean_test_MSE'] = -results['mean_test_neg_mean_squared_error']
results['mean_test_RMSE'] = -results['mean_test_neg_root_mean_squared_error']
results.loc[grid_search.best_index_][['params', 'mean_test_MAE', 'mean_test_MSE', 'mean_test_RMSE', 'mean_test_r2']]


Best Parameters
{'model__algorithm': 'auto', 'model__leaf_size': 10, 'model__metric': 'manhattan', 'model__n_neighbors': 5, 'model__weights': 'distance'}

Best Cross-Validation Accuracy
-6055.69051493398


params            {'model__algorithm': 'auto', 'model__leaf_size...
mean_test_MAE                                             36.649955
mean_test_MSE                                           6055.690515
mean_test_RMSE                                            77.806447
mean_test_r2                                                0.42804
Name: 21, dtype: object

In [6]:
KNN_model = grid_search.best_estimator_
y_pred = KNN_model.predict(X_test)
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

MAE:  35.07565836019646
MSE:  5518.745043419068
RMSE:  74.28825643006482
